<a href="https://colab.research.google.com/github/seankh06/Flyrank-ML-Engineer-Internship-Starter/blob/main/work/scripts/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seankh06/Flyrank-ML-Engineer-Internship-Starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

This is a classification task. I'm trying to predict whether an article falls into a "high engagement" or "low engagement" category, based on its features, mainly word_count. This fits classification rather than the other task types because I already have a clear target with two categories, and I want to predict which one a page belongs to. It's not clustering, since I'm not grouping pages without labels, and it's not ranking, since I'm not ordering pages by priority. The goal here is straightforward: given what we know about a page, predict its
engagement category.

## 2. Target or proxy

The target I'm using is high_engagement, a column I created from sessions_90d. Pages above the median session count are labeled 1 (high engagement), and pages below are labeled 0 (low engagement).

This is a proxy rather than a direct measurement, since sessions_90d doesn't really tell us if a page is genuinely engaging, it just tells us how many times people visited. I chose sessions_90d over ctr because in my earlier discovery, word_count had a moderate correlation with sessions_90d (0.332), while its correlation with ctr was close to zero (-0.119). That made sessions_90d the more realistic target to actually try predicting something meaningful.

## 3. Success metric

The metric I'll use is accuracy, the percentage of pages where the model correctly predicts high or low engagement. This makes sense here because my target is fairly balanced (about 53% high engagement, 47% low engagement), so accuracy won't be misleading the way it would be with
a very imbalanced target. A model that's just guessing randomly would land around 50% accuracy, so anything meaningfully above that would show the model found a real pattern.

## 4. The unit of analysis, as a real dataframe

One row represents one content page or article. Each row shows its word count, its session count over the last 90 days, and the engagement label derived from that.

In [ ]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

median_sessions = df["sessions_90d"].median()
df["high_engagement"] = (df["sessions_90d"] >= median_sessions).astype(int)

df[["content_id", "word_count", "sessions_90d", "high_engagement"]].head(10)

,content_id,word_count,sessions_90d,high_engagement
0,content_304f48230142,3221.0,17,1
1,content_a1fb4e703a9e,2481.0,9,1
2,content_9aa793d4d895,3515.0,11,1
3,content_331d6c4de07b,NaN,78,1
4,content_d99b7a2d90ca,2803.0,145,1
5,content_d4084a4bc775,3080.0,5,0
6,content_9a34b442b552,3059.0,1,0
7,content_a63219c6e95a,NaN,28,1
8,content_5e6c160719bc,3807.0,68,1
9,content_c27558df2b0c,NaN,3,0


## 5. Why ML beats a fixed rule here

A simple if-statement, like "word_count above 2000 means high engagement," only looks at one factor at a time. But engagement is probably influenced by more than just length, things like content_type, avg_position, or freshness could all play a role too. A fixed rule can't combine multiple factors the way a model can. This also lines up with what I found earlier: the correlation between word_count and sessions_90d was only 0.332, a moderate relationship, not a strong one. That suggests the pattern isn't simple or linear enough for one clean rule to capture it reliably. A model can weigh several features together and find patterns a single
if-statement would miss.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.